# Day 27: Football Player Clustering by Style of Play

**Phase 2: EDA & Feature Engineering**

## Tactical & Statistical Overview
In modern sports analytics, clustering algorithms enable **unsupervised archetype discovery**. Instead of relying on nominal line-up labels (e.g. "CM" or "CAM"), we cluster players based on normalized on-ball and off-ball behaviors.

### Roadmap:
1. **Step 1: Position Sample Selection & Feature Engineering:** Load qualified World Cup 2022 midfielders ($\ge 200$ minutes) and select 8 orthogonal style features.
2. **Step 2: Robust Scaling & Dimensionality Reduction:** Neutralize spatial/statistical outliers via `RobustScaler` and extract principal tactical axes via `PCA`.
3. **Step 3: Cluster Optimization (K-Means & HDBSCAN):** Compare centroid partition (Elbow & Silhouette) with density-based clustering for noise/hybrid detection.
4. **Step 4: Tactical Archetype Interpretation & Editorial Visualization:** Name the clusters as real tactical roles and plot high-impact 2D projection and profile comparisons.

## Step 1: Position Sample Selection & Style Features

### Why cluster within a single position group?
- Clustering all outfield positions together yields trivial separations (e.g. Center Backs vs. Strikers).
- The high-value tactical insight lies in discovering **intra-positional archetypes**: within Midfielders, who is a *Deep-Lying Playmaker*, a *Box-to-Box Engine*, a *Ball Winner*, or an *Advanced Creator*?

### Feature Selection (8 Orthogonal Style Metrics):
1. `passes_completed_p90` - Volume of circulation
2. `progressive_passes_p90` - Vertical field progression
3. `key_passes_p90` - Final-third shot creation
4. `xg_p90` - Quality of penalty box arrival & scoring threat
5. `dribbles_completed_p90` - 1v1 take-ons & progressive carries
6. `ball_recoveries_p90` - Defensive possession reclamation
7. `interceptions_p90` - Spatial reading & passing lane disruption
8. `pressures_p90` - Off-ball defensive work rate

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import RobustScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans, HDBSCAN
from sklearn.metrics import silhouette_score
from IPython.display import display

# Visualization styling (Dark editorial theme)
plt.style.use("dark_background")
plt.rcParams["font.sans-serif"] = "DejaVu Sans"
plt.rcParams["font.family"] = "sans-serif"

# 1. Load precomputed per-90 metrics (from Day 23 feature store)
data_dir = Path("../../data")
metrics_path = data_dir / "processed" / "football" / "player_per90_metrics.parquet"
df_players = pd.read_parquet(metrics_path)

# 2. Filter qualified Midfielders (>= 200 minutes)
mids_df = df_players[
    (df_players["position_group"] == "MID") &
    (df_players["total_minutes_played"] >= 200.0)
].copy().reset_index(drop=True)

# 3. Selected Style Features
style_features = [
    "passes_completed_p90",
    "progressive_passes_p90",
    "key_passes_p90",
    "xg_p90",
    "dribbles_completed_p90",
    "ball_recoveries_p90",
    "interceptions_p90",
    "pressures_p90"
]

print(f"Dataset prepared: {len(mids_df)} qualified midfielders across {mids_df['team_name'].nunique()} national teams")
display(mids_df[["player_name", "team_name", "primary_position", "total_minutes_played"] + style_features[:4]].head(10))


## Step 2: Robust Scaling & Dimensionality Reduction (PCA)

### 1. Why `RobustScaler` over `StandardScaler`?
Soccer per-90 metrics exhibit severe positive skewness and elite tactical outliers (e.g. Rodri's passing volume or Mbappé's dribbles).
- `StandardScaler` calculates the sample mean ($\mu$) and standard deviation ($\sigma$), which are heavily distorted by outliers.
- `RobustScaler` scales based on the median ($Q_2$) and the interquartile range ($IQR = Q_3 - Q_1$):
  $$z = \frac{x - Q_2}{Q_3 - Q_1}$$
  This prevents elite outliers from artificially compressing the rest of the cohort.

### 2. Principal Component Analysis (PCA) & Factor Loadings
We project the 8-dimensional style space onto 2 principal components to evaluate the primary tactical trade-offs.

In [ ]:
# 1. Feature matrix and Robust Scaling
X = mids_df[style_features].values
scaler = RobustScaler()
X_scaled = scaler.fit_transform(X)

# 2. PCA Fitting (2 Components)
pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_scaled)
mids_df["pca_1"] = X_pca[:, 0]
mids_df["pca_2"] = X_pca[:, 1]

var_pc1 = pca.explained_variance_ratio_[0]
var_pc2 = pca.explained_variance_ratio_[1]
print(f"PCA Variance Explained: PC1 = {var_pc1:.1%}, PC2 = {var_pc2:.1%} | Total 2D Variance = {var_pc1 + var_pc2:.1%}")

# 3. Factor Loadings Table
loadings_df = pd.DataFrame(
    pca.components_.T,
    columns=["PC1_Direct_Threat", "PC2_Ball_Progression"],
    index=style_features
)

print("\n=== PCA Factor Loadings (Feature Weights per Axis) ===")
display(loadings_df.round(3))

# 4. Visualize PCA Loadings Heatmap
plt.figure(figsize=(8, 5.5))
sns.heatmap(
    loadings_df,
    annot=True,
    fmt=".3f",
    cmap="coolwarm",
    center=0.0,
    cbar=True,
    linewidths=0.8,
    linecolor="#222222"
)
plt.title("PCA Tactical Factor Loadings (Weights per Feature)", fontsize=13, fontweight="bold", pad=12)
plt.ylabel("Style Feature", fontsize=11)
plt.tight_layout()
plt.show()


## Step 3: Unsupervised Clustering (K-Means Optimization & HDBSCAN)

### 1. K-Means Parameter Selection:
- **Inertia (Elbow Method):** Measures intra-cluster sum of squared distances to centroids.
- **Silhouette Score:** Measures cohesion within clusters compared to separation from neighboring clusters (ranges from -1 to +1).
- We evaluate $k \in [2, 7]$. While $k=2$ splits general attacking vs. defensive orientation, **$k=4$** provides the sweet spot for actionable tactical archetypes.

### 2. Density-Based Clustering (HDBSCAN):
- Unlike K-Means, `HDBSCAN` does not assume spherical clusters or equal sizes.
- Crucially, HDBSCAN assigns **`-1` (Noise)** to points in sparse regions, identifying tactical outliers and "unicorn" hybrid players who do not fit rigid positional templates.

In [ ]:
# 1. K-Means Elbow & Silhouette Evaluation
k_range = range(2, 8)
inertias = []
silhouettes = []

for k in k_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=15)
    labels = km.fit_predict(X_scaled)
    inertias.append(km.inertia_)
    silhouettes.append(silhouette_score(X_scaled, labels))

# 2. Fit Optimal K-Means Model (k=4)
km_optimal = KMeans(n_clusters=4, random_state=42, n_init=25)
mids_df["kmeans_cluster"] = km_optimal.fit_predict(X_scaled)

# 3. Fit HDBSCAN for Outlier/Noise Detection
hdb = HDBSCAN(min_cluster_size=6, min_samples=3)
mids_df["hdbscan_cluster"] = hdb.fit_predict(X_scaled)

noise_count = (mids_df["hdbscan_cluster"] == -1).sum()
print(f"K-Means Silhouette for k=4: {silhouettes[2]:.3f}")
print(f"HDBSCAN identified {len(set(mids_df['hdbscan_cluster']) - {-1})} core dense clusters and {noise_count} hybrid/outlier players (labeled -1)")

# 4. Plot Diagnostic Optimization Curves
fig, ax1 = plt.subplots(figsize=(8, 4.5))
ax1.plot(k_range, silhouettes, 'o-', color="#00d2d3", linewidth=2.5, markersize=8, label="Silhouette Score")
ax1.set_xlabel("Number of Clusters (k)", fontsize=11)
ax1.set_ylabel("Silhouette Score", color="#00d2d3", fontsize=11)
ax1.set_title("K-Means Optimization: Silhouette & Elbow Curves", fontsize=13, fontweight="bold", pad=12)
ax1.grid(alpha=0.2)

ax2 = ax1.twinx()
ax2.plot(k_range, inertias, 's--', color="#ff9f43", linewidth=2, markersize=7, label="Inertia (Elbow)")
ax2.set_ylabel("Inertia (Sum of Squared Distances)", color="#ff9f43", fontsize=11)

plt.tight_layout()
plt.show()


## Step 4: Tactical Archetype Profiling & Editorial Visualizations

### Mapping Clusters to Real Tactical Roles:
Based on centroid analysis and per-90 metrics, we name the 4 discovered archetypes:
1. **Cluster 0: Box-to-Box & Pressing Engine (N=37):**
   - Leading in defensive pressure work rate (`pressures_p90 = 16.4`) and ball recoveries (`4.90 p90`).
   - Exemplars: Adrien Rabiot, André-Frank Zambo Anguissa, Idrissa Gueye, Aaron Ramsey.
2. **Cluster 1: Deep-Lying Playmaker & Circulator (N=30):**
   - Dominates passing volume (`54.0 passes completed p90`) and vertical field progression (`10.1 progressive passes p90`).
   - Exemplars: Christian Eriksen, Kevin De Bruyne, Bernardo Silva, Aaron Mooy, Granit Xhaka, Rodri.
3. **Cluster 2: Direct Dribbler & High-Threat Wing Creator (N=13):**
   - Exceptional 1v1 take-ons (`2.90 dribbles p90`), scoring arrival (`0.39 xG p90`), and shot assistance (`1.64 key passes p90`).
   - Exemplars: Kylian Mbappé, Ángel Di María, Neymar Jr, Phil Foden, Ismaïla Sarr.
4. **Cluster 3: Wide Attacking Link & Supporting Midfielder (N=24):**
   - Balanced transition connector with moderate key passes (`1.08 p90`) and selective box entries.
   - Exemplars: Son Heung-min, André Ayew, Timothy Weah, Jordan Henderson, Xherdan Shaqiri.

In [ ]:
# 1. Map Archetype Labels
archetype_map = {
    0: "Box-to-Box & Pressing Engine",
    1: "Deep-Lying Playmaker & Circulator",
    2: "Direct Dribbler & Goal Threat",
    3: "Wide Attacking Link & Supporting Mid"
}
mids_df["archetype"] = mids_df["kmeans_cluster"].map(archetype_map)

# Palette configuration
archetype_palette = {
    "Box-to-Box & Pressing Engine": "#54a0ff",
    "Deep-Lying Playmaker & Circulator": "#1dd1a1",
    "Direct Dribbler & Goal Threat": "#ff6b6b",
    "Wide Attacking Link & Supporting Mid": "#feca57"
}

# 2. Figure 1: 2D Tactical Landscape with Player Annotations
plt.figure(figsize=(14, 8.5))
ax = sns.scatterplot(
    data=mids_df,
    x="pca_1", y="pca_2",
    hue="archetype",
    palette=archetype_palette,
    s=120,
    alpha=0.88,
    edgecolor="white",
    linewidth=0.8
)

# Annotate marquee players across clusters
marquee_players = [
    ("Kylian Mbappé Lottin", "Mbappé"),
    ("Kevin De Bruyne", "De Bruyne"),
    ("Christian Dannemann Eriksen", "Eriksen"),
    ("Adrien Rabiot", "Rabiot"),
    ("Heung-Min Son", "Son"),
    ("Ángel Fabián Di María Hernández", "Di María"),
    ("Bernardo Mota Veiga de Carvalho e Silva", "Bernardo Silva"),
    ("Idrissa Gana Gueye", "Gueye"),
    ("André-Frank Zambo Anguissa", "Zambo Anguissa")
]

for p_full, p_short in marquee_players:
    p_sub = mids_df[mids_df["player_name"] == p_full]
    if not p_sub.empty:
        p_row = p_sub.iloc[0]
        ax.annotate(
            p_short,
            (p_row["pca_1"], p_row["pca_2"]),
            xytext=(6, 6), textcoords="offset points",
            fontsize=9, color="white", fontweight="bold",
            bbox=dict(boxstyle="round,pad=0.25", facecolor="#1a1f2c", alpha=0.85, edgecolor="#666666")
        )

plt.title("World Cup 2022: Midfielder Tactical Style Space (K-Means k=4 Discovered Archetypes)", fontsize=14, fontweight="bold", pad=12)
plt.xlabel("PC1: Direct Attacking Threat & 1v1 vs. Deep Circulation (31.9% Variance)", fontsize=11)
plt.ylabel("PC2: Proactive Possession Volume & Field Progression (18.8% Variance)", fontsize=11)
plt.legend(loc="upper right", facecolor="#1a1f2c", edgecolor="#444444", fontsize=9.5)
plt.grid(alpha=0.15)
plt.tight_layout()
plt.show()

# 3. Figure 2: Standardized Centroid Profile Heatmap
scaled_features_df = pd.DataFrame(X_scaled, columns=style_features)
scaled_features_df["archetype"] = mids_df["archetype"]
centroid_profiles = scaled_features_df.groupby("archetype").mean()

plt.figure(figsize=(11, 5.5))
sns.heatmap(
    centroid_profiles,
    annot=True,
    fmt=".2f",
    cmap="vlag",
    center=0.0,
    linewidths=1.0,
    linecolor="#1a1f2c"
)
plt.title("Discovered Midfielder Archetypes: Standardized Feature Profiles (RobustScaler Medians)", fontsize=13, fontweight="bold", pad=12)
plt.ylabel("Tactical Archetype", fontsize=11)
plt.xlabel("Style Feature", fontsize=11)
plt.xticks(rotation=25, ha="right", fontsize=9.5)
plt.tight_layout()
plt.show()


## Summary & Portfolio Takeaways (Day 27)

1. **Position-Specific Clustering:**
   - By narrowing our scope to Midfielders with $\ge 200$ minutes, we bypassed trivial positional divides to reveal nuanced functional archetypes.

2. **The Role of RobustScaler & PCA:**
   - Normalizing by the interquartile range prevented high-volume passing outliers from flattening creative metrics.
   - PCA captured over 50.7% of style variance in 2 orthogonal dimensions, separating direct vertical runners from tempo controllers.

3. **K-Means vs. HDBSCAN in Scouting:**
   - K-Means cleanly partitions the market into 4 interpretable tactical categories for roster balancing.
   - HDBSCAN flags unclassifiable "noise" players (`-1`), highlighting versatile hybrids and singular tactical profiles.